# Feature Engineering

In this notebook, we create additional informative features from the preprocessed dataset.

The objective is to improve the predictive power of the machine learning models by generating meaningful features while avoiding any data leakage.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

BASE_PATH = os.getenv("BASE_PATH")

processed_path = Path(BASE_PATH) / "data" / "processed"

X_train = pd.read_parquet(processed_path / "X_train.parquet")
X_test = pd.read_parquet(processed_path / "X_test.parquet")

y_train = pd.read_parquet(processed_path / "y_train.parquet")
y_test = pd.read_parquet(processed_path / "y_test.parquet")

In [3]:
print(X_train.shape)
print(X_test.shape)

X_train.head()

(79588, 112)
(19898, 112)


,MinTemp,MaxTemp,Rainfall,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
24209,24.100000,34.400002,0.000000,59.0,28.0,26.0,30.0,37.0,1002.500000,1002.799988,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12518,19.900000,22.799999,17.799999,61.0,37.0,33.0,81.0,78.0,1022.799988,1023.200012,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
82314,6.400000,22.200001,0.000000,50.0,13.0,28.0,51.0,83.0,1011.700012,1009.200012,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
60027,23.200001,30.400000,0.000000,39.0,20.0,28.0,74.0,64.0,1011.700012,1008.500000,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4058,17.400000,26.600000,0.400000,28.0,4.0,17.0,87.0,53.0,1019.599976,1017.099976,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Feature Engineering Strategy

In this notebook we will engineer new weather-related features based on existing meteorological variables.

These features aim to capture hidden relationships that may improve the model performance.

## Temperature Range

Difference between the maximum and minimum temperature.

In [4]:
for df in [X_train, X_test]:
    df["TempRange"] = df["MaxTemp"] - df["MinTemp"]

X_train[["MinTemp","MaxTemp","TempRange"]].head()

,MinTemp,MaxTemp,TempRange
24209,24.100000,34.400002,10.300001
12518,19.900000,22.799999,2.900000
82314,6.400000,22.200001,15.800001
60027,23.200001,30.400000,7.199999
4058,17.400000,26.600000,9.200001


## Temperature Change

Difference between the afternoon and morning temperatures.

In [5]:
for df in [X_train, X_test]:
    df["TempChange"] = df["Temp3pm"] - df["Temp9am"]

X_train[["Temp9am","Temp3pm","TempChange"]].head()

,Temp9am,Temp3pm,TempChange
24209,29.799999,29.100000,-0.699999
12518,21.200001,20.799999,-0.400002
82314,19.000000,17.500000,-1.500000
60027,27.299999,29.100000,1.800001
4058,19.000000,25.900000,6.900000


## Temperature Statistics

Create average temperature from the morning and afternoon temperatures.

In [6]:
for df in [X_train, X_test]:
    df["AverageTemp"] = (
        df["Temp9am"] + df["Temp3pm"]
    ) / 2

X_train[
    [
        "Temp9am",
        "Temp3pm",
        "AverageTemp"
    ]
].head()

,Temp9am,Temp3pm,AverageTemp
24209,29.799999,29.100000,29.450001
12518,21.200001,20.799999,21.000000
82314,19.000000,17.500000,18.250000
60027,27.299999,29.100000,28.200001
4058,19.000000,25.900000,22.450001


In [8]:
for df in [X_train, X_test]: 
    df['HumidityChange'] = df['Humidity3pm'] - df['Humidity9am']
X_train[["Humidity9am","Humidity3pm","HumidityChange"]].head()

,Humidity9am,Humidity3pm,HumidityChange
24209,30.0,37.0,7.0
12518,81.0,78.0,-3.0
82314,51.0,83.0,32.0
60027,74.0,64.0,-10.0
4058,87.0,53.0,-34.0


In [10]:
for df in [X_train, X_test]: 
    df['AverageHumidity'] = (df['Humidity9am'] + df['Humidity3pm']) / 2


X_train[['Humidity9am', 'Humidity3pm', 'AverageHumidity']].head()

,Humidity9am,Humidity3pm,AverageHumidity
24209,30.0,37.0,33.5
12518,81.0,78.0,79.5
82314,51.0,83.0,67.0
60027,74.0,64.0,69.0
4058,87.0,53.0,70.0


## Temperature-Humidity Interaction

Create an interaction feature between average temperature and average humidity.

In [11]:
for df in [X_train, X_test]:
    df["TempHumidityInteraction"] = (
        df["AverageTemp"] * df["AverageHumidity"]
    )

X_train[
    [
        "AverageTemp",
        "AverageHumidity",
        "TempHumidityInteraction"
    ]
].head()

,AverageTemp,AverageHumidity,TempHumidityInteraction
24209,29.450001,33.5,986.575012
12518,21.000000,79.5,1669.500000
82314,18.250000,67.0,1222.750000
60027,28.200001,69.0,1945.800049
4058,22.450001,70.0,1571.500000


## Pressure Difference

Difference between afternoon and morning atmospheric pressure.

In [ ]:
for df in [X_train, X_test]:
    df["PressureDiff"] = df["Pressure3pm"] - df["Pressure9am"]

X_train[["Pressure9am","Pressure3pm","PressureDiff"]].head()

,Pressure9am,Pressure3pm,PressureDiff
90330,1004.299988,1006.700012,2.400024
11065,1017.700012,1015.299988,-2.400024
61933,1014.700012,1012.500000,-2.200012
63928,1000.000000,996.599976,-3.400024
40007,1019.599976,1016.900024,-2.699951


In [12]:
for df in [X_train, X_test]: 
    df['AveragePressure'] = (df['Pressure9am'] + df['Pressure3pm']) / 2

X_train[['Pressure9am', 'Pressure3pm', 'AveragePressure']].head()

,Pressure9am,Pressure3pm,AveragePressure
24209,1002.500000,1002.799988,1002.650024
12518,1022.799988,1023.200012,1023.000000
82314,1011.700012,1009.200012,1010.450012
60027,1011.700012,1008.500000,1010.099976
4058,1019.599976,1017.099976,1018.349976


## Humidity Difference

Difference between afternoon and morning humidity.

In [ ]:
for df in [X_train, X_test]:
    df["HumidityDiff"] = df["Humidity3pm"] - df["Humidity9am"]

X_train[["Humidity9am","Humidity3pm","HumidityDiff"]].head()

,Humidity9am,Humidity3pm,HumidityDiff
90330,63.0,31.0,-32.0
11065,66.0,39.0,-27.0
61933,54.0,59.0,5.0
63928,82.0,69.0,-13.0
40007,83.0,44.0,-39.0


## Wind Speed Difference

Difference between afternoon and morning wind speed.

In [ ]:
for df in [X_train, X_test]:
    df["WindSpeedDiff"] = df["WindSpeed3pm"] - df["WindSpeed9am"]

X_train[["WindSpeed9am","WindSpeed3pm","WindSpeedDiff"]].head()

,WindSpeed9am,WindSpeed3pm,WindSpeedDiff
90330,9.0,22.0,13.0
11065,6.0,15.0,9.0
61933,13.0,31.0,18.0
63928,24.0,24.0,0.0
40007,17.0,7.0,-10.0


## Average Wind Speed

The average wind speed combines morning and afternoon wind measurements.

In [ ]:
for df in [X_train, X_test]:
    df["AverageWindSpeed"] = (
        df["WindSpeed9am"] + df["WindSpeed3pm"]
    ) / 2

X_train[
    ["WindSpeed9am", "WindSpeed3pm", "AverageWindSpeed"]
].head()

,WindSpeed9am,WindSpeed3pm,AverageWindSpeed
90330,9.0,22.0,15.5
11065,6.0,15.0,10.5
61933,13.0,31.0,22.0
63928,24.0,24.0,24.0
40007,17.0,7.0,12.0


## Display Engineered Features

In [ ]:
engineered_cols = [
    "TempRange",
    "TempChange",
    "PressureDiff",
    "HumidityDiff",
    "WindSpeedDiff"
]

X_train[engineered_cols].head()

,TempRange,TempChange,PressureDiff,HumidityDiff,WindSpeedDiff
90330,0.0,4.700001,2.400024,-32.0,13.0
11065,0.0,5.900002,-2.400024,-27.0,9.0
61933,0.0,0.200001,-2.200012,5.0,18.0
63928,0.0,2.200001,-3.400024,-13.0,0.0
40007,0.0,9.500000,-2.699951,-39.0,-10.0


## Dataset Information

In [ ]:
print(X_train.shape)
X_train.info()

(79587, 117)
<class 'pandas.DataFrame'>
Index: 79587 entries, 90330 to 2591
Columns: 117 entries, MinTemp to WindSpeedDiff
dtypes: Int8(2), float32(17), float64(97), int64(1)
memory usage: 65.6 MB


## Pressure-Wind Interaction

This feature combines pressure and wind speed information into a single predictor.

In [ ]:
for df in [X_train, X_test]:
    df["PressureWindInteraction"] = (
        df["AveragePressure"] * df["AverageWindSpeed"]
    )

X_train[
    [
        "AveragePressure",
        "AverageWindSpeed",
        "PressureWindInteraction",
    ]
].head()

,AveragePressure,AverageWindSpeed,PressureWindInteraction
90330,1005.500000,15.5,15585.250000
11065,1016.500000,10.5,10673.250000
61933,1013.599976,22.0,22299.199219
63928,998.299988,24.0,23959.199219
40007,1018.250000,12.0,12219.000000


## Rainfall Log Transformation

Rainfall is highly skewed. A logarithmic transformation helps reduce the influence of extreme rainfall values.

In [ ]:
for df in [X_train, X_test]:
    df["RainfallLog"] = np.log1p(df["Rainfall"])

X_train[
    ["Rainfall", "RainfallLog"]
].head()

,Rainfall,RainfallLog
90330,1.400000,0.875469
11065,0.000000,0.000000
61933,0.000000,0.000000
63928,77.800003,4.366913
40007,1.000000,0.693147


## Final Dataset Information

In [ ]:
print(X_train.shape)
print(X_test.shape)

X_train.info()

(79587, 124)
(19897, 124)
<class 'pandas.DataFrame'>
Index: 79587 entries, 90330 to 2591
Columns: 124 entries, MinTemp to RainfallLog
dtypes: Int8(2), float32(24), float64(97), int64(1)
memory usage: 67.7 MB


In [ ]:
X_train.shape
X_test.shape

(19897, 124)

In [ ]:
X_train.columns.tolist()

['MinTemp',
 'MaxTemp',
 'Rainfall',
 'WindGustSpeed',
 'WindSpeed9am',
 'WindSpeed3pm',
 'Humidity9am',
 'Humidity3pm',
 'Pressure9am',
 'Pressure3pm',
 'Cloud9am',
 'Cloud3pm',
 'Temp9am',
 'Temp3pm',
 'RainToday',
 'Location_Adelaide',
 'Location_Albany',
 'Location_Albury',
 'Location_AliceSprings',
 'Location_BadgerysCreek',
 'Location_Ballarat',
 'Location_Bendigo',
 'Location_Brisbane',
 'Location_Cairns',
 'Location_Canberra',
 'Location_Cobar',
 'Location_CoffsHarbour',
 'Location_Dartmoor',
 'Location_Darwin',
 'Location_GoldCoast',
 'Location_Hobart',
 'Location_Katherine',
 'Location_Launceston',
 'Location_Melbourne',
 'Location_MelbourneAirport',
 'Location_Mildura',
 'Location_Moree',
 'Location_MountGambier',
 'Location_MountGinini',
 'Location_Newcastle',
 'Location_Nhil',
 'Location_NorahHead',
 'Location_NorfolkIsland',
 'Location_Nuriootpa',
 'Location_PearceRAAF',
 'Location_Penrith',
 'Location_Perth',
 'Location_PerthAirport',
 'Location_Portland',
 'Location_Ric

In [ ]:
len(X_train.columns)

124

In [ ]:
X_train.isna().sum().sort_values(ascending=False).head(20)

MinTemp                   0
MaxTemp                   0
Rainfall                  0
WindGustSpeed             0
WindSpeed9am              0
WindSpeed3pm              0
Humidity9am               0
Humidity3pm               0
Pressure9am               0
Pressure3pm               0
Cloud9am                  0
Cloud3pm                  0
Temp9am                   0
Temp3pm                   0
RainToday                 0
Location_Adelaide         0
Location_Albany           0
Location_Albury           0
Location_AliceSprings     0
Location_BadgerysCreek    0
dtype: int64

In [ ]:
X_train.head()

,MinTemp,MaxTemp,Rainfall,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,...,PressureDiff,HumidityDiff,WindSpeedDiff,AverageTemp,AverageHumidity,AveragePressure,AverageWindSpeed,TempHumidityInteraction,PressureWindInteraction,RainfallLog
90330,16.700001,16.700001,1.400000,74.0,9.0,22.0,63.0,31.0,1004.299988,1006.700012,...,2.400024,-32.0,13.0,16.700001,47.0,1005.500000,15.5,784.900024,15585.250000,0.875469
11065,33.200001,33.200001,0.000000,39.0,6.0,15.0,66.0,39.0,1017.700012,1015.299988,...,-2.400024,-27.0,9.0,33.200001,52.5,1016.500000,10.5,1743.000000,10673.250000,0.000000
61933,27.900000,27.900000,0.000000,43.0,13.0,31.0,54.0,59.0,1014.700012,1012.500000,...,-2.200012,5.0,18.0,27.900000,56.5,1013.599976,22.0,1576.349976,22299.199219,0.000000
63928,31.700001,31.700001,77.800003,65.0,24.0,24.0,82.0,69.0,1000.000000,996.599976,...,-3.400024,-13.0,0.0,31.700001,75.5,998.299988,24.0,2393.350098,23959.199219,4.366913
40007,27.600000,27.600000,1.000000,39.0,17.0,7.0,83.0,44.0,1019.599976,1016.900024,...,-2.699951,-39.0,-10.0,27.600000,63.5,1018.250000,12.0,1752.599976,12219.000000,0.693147


In [ ]:
y_train.value_counts()

RainTomorrow
False           61702
True            17885
Name: count, dtype: int64

In [ ]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(79587, 124)
(19897, 124)
(79587, 1)
(19897, 1)


In [16]:
(X_train["MinTemp"] == X_train["MaxTemp"]).sum()

np.int64(7)

In [ ]:
X_train.loc[
    X_train["MinTemp"] == X_train["MaxTemp"],
    ["MinTemp", "MaxTemp", "AverageTemp", "TempHumidityInteraction"]
].head(20)

,MinTemp,MaxTemp,AverageTemp,TempHumidityInteraction
90330,16.700001,16.700001,16.700001,784.900024
11065,33.200001,33.200001,33.200001,1743.000000
61933,27.900000,27.900000,27.900000,1576.349976
63928,31.700001,31.700001,31.700001,2393.350098
40007,27.600000,27.600000,27.600000,1752.599976
60587,24.400000,24.400000,24.400000,1512.799927
66691,14.000000,14.000000,14.000000,1183.000000
66967,19.400000,19.400000,19.400000,1173.699951
31930,13.100000,13.100000,13.100000,818.750000
8147,22.000000,22.000000,22.000000,1650.000000


In [18]:
(X_train["MinTemp"] == X_train["MaxTemp"]).value_counts()

False    79581
True         7
Name: count, dtype: int64

In [ ]:
print(X_train.shape)
print(X_test.shape)

print()

print(X_train.columns[-15:].tolist())

(79587, 124)
(19897, 124)

['WindDir3pm_W', 'WindDir3pm_WNW', 'WindDir3pm_WSW', 'TempRange', 'TempChange', 'PressureDiff', 'HumidityDiff', 'WindSpeedDiff', 'AverageTemp', 'AverageHumidity', 'AveragePressure', 'AverageWindSpeed', 'TempHumidityInteraction', 'PressureWindInteraction', 'RainfallLog']


In [ ]:
X_train[
    [
        "AverageTemp",
        "AverageHumidity",
        "TempHumidityInteraction",
        "RainfallLog",
        "PressureWindInteraction"
    ]
].head()

,AverageTemp,AverageHumidity,TempHumidityInteraction,RainfallLog,PressureWindInteraction
90330,13.750000,47.0,646.250000,0.875469,15585.250000
11065,29.950001,52.5,1572.375000,0.000000,10673.250000
61933,23.400000,56.5,1322.099976,0.000000,22299.199219
63928,29.600000,75.5,2234.800049,4.366913,23959.199219
40007,22.350000,63.5,1419.224976,0.693147,12219.000000


In [19]:
X_train.dtypes.value_counts()

float64    97
float32    19
Int8        2
int64       1
Name: count, dtype: int64

In [20]:
X_train.info(verbose=True)

<class 'pandas.DataFrame'>
Index: 79588 entries, 24209 to 23534
Data columns (total 119 columns):
 #    Column                     Dtype  
---   ------                     -----  
 0    MinTemp                    float32
 1    MaxTemp                    float32
 2    Rainfall                   float32
 3    WindGustSpeed              float32
 4    WindSpeed9am               float32
 5    WindSpeed3pm               float32
 6    Humidity9am                float32
 7    Humidity3pm                float32
 8    Pressure9am                float32
 9    Pressure3pm                float32
 10   Cloud9am                   Int8   
 11   Cloud3pm                   Int8   
 12   Temp9am                    float32
 13   Temp3pm                    float32
 14   RainToday                  int64  
 15   Location_Adelaide          float64
 16   Location_Albany            float64
 17   Location_Albury            float64
 18   Location_AliceSprings      float64
 19   Location_BadgerysCreek     float64


In [30]:
for df in [X_train, X_test]:
    numeric_cols = df.select_dtypes(include='float64').columns
    df[numeric_cols] = df[numeric_cols].astype('float32') # type: ignore
    df['RainToday'] = df['RainToday'].astype('uint8')   # type: ignore

In [31]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 79588 entries, 24209 to 23534
Columns: 119 entries, MinTemp to AveragePressure
dtypes: Int8(2), float32(19), uint8(98)
memory usage: 14.1 MB
